## Loading The Data
We start by loading the 20 newsgroups dataset's raw text data, and split it into train and test.

In [44]:
from sklearn.datasets import fetch_20newsgroups

# Load the full training and testing dataset.
data_train = fetch_20newsgroups(subset='train', shuffle=True, random_state=42)
data_test = fetch_20newsgroups(subset='test', shuffle=True, random_state=42)

# Get the raw documents and labels.
X_train, y_train = data_train.data, data_train.target
X_test, y_test = data_test.data, data_test.target

# Target names (class labels):
target_names = data_train.target_names

print('Training data samples:', len(X_train))
print('Testing data samples:', len(X_test))
print('Number of classes:', len(target_names))

Training data samples: 11314
Testing data samples: 7532
Number of classes: 20


## Vectorizing The Data
We will vectorize the data in two ways, 1. TF-IDF 2.Count Vectorizer
Count Vectorizer use raw counts of words and TF-IDF gives a score based on importance, let's see the difference with a random document example.

In [53]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

import numpy as np
import pandas as pd

# --------------------
# 1. Fit Vectorizers
# --------------------
count_vect = CountVectorizer()
X_train_count = count_vect.fit_transform(X_train)
X_test_count = count_vect.transform(X_test)

tfidf_vect = TfidfVectorizer()
X_train_tfidf = tfidf_vect.fit_transform(X_train)
X_test_tfidf = tfidf_vect.transform(X_test)


# --------------------
# 2. Pick a Single Document
# --------------------
doc_index = 0 # any document index from 0 to 11313 
doc_text = X_train[doc_index]

# --------------------
# 3. Extract the Vector for That Document
# --------------------
doc_count_vec = X_train_count[doc_index]
doc_tfidf_vec = X_train_tfidf[doc_index]

doc_count_arr = doc_count_vec.toarray().flatten() # Convert to dense array and flatten
doc_tfidf_arr = doc_tfidf_vec.toarray().flatten() # Convert to dense array and flatten

# Get feature names
count_features = count_vect.get_feature_names_out()
tfidf_features = tfidf_vect.get_feature_names_out()

# --------------------
# 4. Find Nonzero Elements
# --------------------
nonzero_count_idx = doc_count_arr.nonzero()[0]    # indices of nonzero tokens (Count)
nonzero_tfidf_idx = doc_tfidf_arr.nonzero()[0]    # indices of nonzero tokens (TF-IDF)


# --------------------
# 5. Combine into a Single DataFrame
# --------------------

# For a fair comparison, let's focus on words that appear at least once in this document,
# which means the union of nonzero_count_idx and nonzero_tfidf_idx.
nonzero_union = np.union1d(nonzero_count_idx, nonzero_tfidf_idx)

features = [count_features[i] for i in nonzero_union]
counts = doc_count_arr[nonzero_union]
tfidfs = doc_tfidf_arr[nonzero_union]

# 'Count' - Number of times the token(word) appears in the document
# 'TF-IDF' - TF-IDF score of the token in the document
df = pd.DataFrame({
    'token': features,
    'count': counts,
    'tfidf': tfidfs
})

# Sort by count or tfidf (descending)
df_sorted = df.sort_values(by='tfidf', ascending=False)

# --------------------
# 6. Display the Comparison
# --------------------
print("\n=== TOKEN COUNTS vs. TF-IDF (Single Document) ===\n")
print(df_sorted.head(70))  # Show top 20 tokens



=== TOKEN COUNTS vs. TF-IDF (Single Document) ===

     token  count     tfidf
14     car      5  0.381339
38  lerxst      2  0.353835
79     wam      2  0.259709
77     umd      2  0.211868
70  tellme      1  0.176918
..     ...    ...       ...
81    were      1  0.049438
55   other      1  0.045496
36    know      1  0.042808
88    your      1  0.042785
76      to      2  0.042473

[70 rows x 3 columns]


# Classifiers Performance Analysis

Instead of manually transforming and training the models separately, we can define a pipeline that does it all in one go.

## Comparing Classifiers
### We will compare 4 different classifiers on the data:
- Logistic Regression
- Decision Tree
- kNN
- SVM

### Let's see an example of a document from our data

In [ ]:
print('Sample document:')
print(X_train[0])